In [4]:
from dotenv import load_dotenv
load_dotenv()

True

In [21]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_ollama import OllamaEmbeddings
from langchain_chroma import Chroma
from langchain.agents import create_agent
from langchain.tools import tool
from langgraph.checkpoint.memory import InMemorySaver
from langchain_groq import ChatGroq


### LOADING THE DOCUMENT

In [6]:
loader = PyPDFLoader("M:\\Agentic-RAG\\Data\\LLM.pdf")
documents = loader.load()
len(documents)

10

### MAKING DOCS INTO CHUNKS

In [7]:
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
chunks = splitter.split_documents(documents)
len(chunks)

18

### creating an embedding model

In [8]:
embeddings = OllamaEmbeddings(model="nomic-embed-text:latest")

### creating a vector store and store the embeddings

In [9]:
vector_store = Chroma.from_documents(
   documents=chunks, 
   embedding=embeddings,
   )

In [11]:
similar = vector_store.similarity_search("What is the difference between a language model and a large language model?", k=3)

### creating a tool

In [18]:
@tool
def retrieve_answer(query):
    """Retrieves the answer to a query from the vector store."""
    similar_docs = vector_store.similarity_search(query, k=3)
    for doc in similar_docs:
        return doc.page_content

### in memory svaer for agent

In [20]:
memory = InMemorySaver()

### llm -> chat groq

In [22]:
llm = ChatGroq(
   model = "openai/gpt-oss-120b",
   temperature = 0.7,
)

### creating an agent

In [25]:
agent = create_agent(
   model = llm,
   tools = [retrieve_answer],
   checkpointer = memory,
   system_prompt = "You are a helpful assistant that retrieves information from a PDF document to answer questions.",
)

In [30]:
query = input("User: ")

In [ ]:
response = agent.invoke({"messages": [{"role": "user", "content": query}]},
             {"configurable": {"thread_id": "user-123"}}
              )

In [31]:
print("Ai: ",response["messages"][-1].content)

Ai:  A **language model (LM)** is any statistical or neural model that’s been trained to predict the next word (or token) in a sequence of text. Its job is to learn the patterns of a language so it can generate or evaluate text, translate, summarize, etc. The size of the model—how many parameters it has, how much data it was trained on, and how much compute was used—can vary widely. Small LMs might have a few million to a few hundred million parameters and are often built for narrow, specialized tasks.

A **large language model (LLM)** is simply a language model that is **much larger** in scale:

| Aspect | Language Model (general) | Large Language Model |
|--------|--------------------------|----------------------|
| **Parameters** | Typically up to a few hundred million (sometimes a few billion) | Usually **billions** of parameters (e.g., 7 B, 70 B, 175 B, etc.) |
| **Training Data** | May be trained on a modest, domain‑specific corpus | Trained on **massive, diverse** text corpora (